# <div align="center"> Statistical Learning </div>
# <div align="center"> Machine Learning and Econometrics </div>
## <div align="center"> ECO 4971/6971 </div>
#### <div align="center">Class 12 — Hands On: Neural Networks with Keras</div>
<div align="center"> Jonathan Holmes (he/him)</div>

## The question

Economists have studied the age–wage relationship for decades. Wages typically rise with experience, peak somewhere in mid-career, and then level off or decline. In Class 9 we modelled this curved shape by adding **polynomial terms by hand** — telling the model to try $\text{age}^2$, $\text{age}^3$, and so on.

Today's question: **can a neural network learn that shape directly from data, without us specifying a functional form?**

By the end of this notebook you will have:
- Seen that a neural network with *no hidden layers* is exactly linear regression.
- Watched the fitted curve bend as a hidden layer is added.
- Connected the training and validation loss curves to the **bias–variance tradeoff** from Class 5.
- Run interactive experiments by adjusting the number of units and layers.
- Compared a neural network against OLS when all predictors are included.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def draw_network(ax, layer_sizes, labels=None, max_nodes=5, title=''):
    """
    Draw a schematic neural network on ax.
    layer_sizes : units per layer including input and output
    labels      : text annotation below each layer column
    max_nodes   : max nodes drawn before showing an ellipsis
    """
    n_layers = len(layer_sizes)
    if labels is None:
        labels = [''] * n_layers

    h_gap, v_gap, r = 3.0, 1.0, 0.28

    def node_color(i):
        if i == 0:          return '#5B9BD5'   # input  — blue
        if i == n_layers-1: return '#ED7D31'   # output — orange
        return '#70AD47'                        # hidden — green

    def visible(size):
        if size <= max_nodes:
            return list(range(size)), False
        return list(range(max_nodes - 1)) + [size - 1], True

    # compute node positions
    all_pos = {}
    for i, size in enumerate(layer_sizes):
        vnodes, _ = visible(size)
        n = len(vnodes)
        ys = [(j - (n - 1) / 2) * v_gap for j in range(n)]
        all_pos[i] = [(i * h_gap, y) for y in ys]

    # edges (behind nodes)
    for i in range(n_layers - 1):
        for (x1, y1) in all_pos[i]:
            for (x2, y2) in all_pos[i + 1]:
                ax.plot([x1, x2], [y1, y2], color='#bbbbbb', lw=0.7, alpha=0.6, zorder=1)

    # nodes
    for i, size in enumerate(layer_sizes):
        _, has_ellipsis = visible(size)
        color = node_color(i)
        ys = [p[1] for p in all_pos[i]]
        x  = all_pos[i][0][0]
        for j, y in enumerate(ys):
            if has_ellipsis and j == len(ys) - 2:
                ax.text(x, (y + ys[j + 1]) / 2, '⋮', ha='center', va='center',
                        fontsize=13, zorder=4)
            ax.add_patch(plt.Circle((x, y), r, color=color, ec='white', lw=1.2, zorder=3))
        y_top, y_bot = max(ys) + r, min(ys) - r
        if size > 1:
            ax.text(x, y_top + 0.18, f'{size} units', ha='center', va='bottom',
                    fontsize=8, color='#888888')
        ax.text(x, y_bot - 0.22, labels[i], ha='center', va='top',
                fontsize=8.5, color='#333333')

    ax.set_title(title, fontsize=11, pad=8)
    ax.set_aspect('equal')
    ax.axis('off')


fig, axes = plt.subplots(1, 3, figsize=(16, 6))

draw_network(axes[0],
    layer_sizes=[1, 1],
    labels=['age\n(input)', 'log wage\n(output)'],
    title='Part 2 — Linear model\n(no hidden layers = OLS)')

draw_network(axes[1],
    layer_sizes=[1, 64, 1],
    labels=['age\n(input)', 'ReLU × 64\n(hidden)', 'log wage\n(output)'],
    title='Part 3 — Shallow network\n(1 hidden layer, 64 units)')

draw_network(axes[2],
    layer_sizes=[1, 64, 64, 1],
    labels=['age\n(input)', 'ReLU × 64\n(hidden 1)', 'ReLU × 64\n(hidden 2)', 'log wage\n(output)'],
    title='Part 5 — Deep network\n(2 hidden layers, 64 units each)')

legend_handles = [
    mpatches.Patch(color='#5B9BD5', label='Input'),
    mpatches.Patch(color='#70AD47', label='Hidden (ReLU)'),
    mpatches.Patch(color='#ED7D31', label='Output'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=3, fontsize=10, frameon=False)
plt.suptitle('The three architectures we will build today', fontsize=13)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

## The data: Wage

The **Wage** dataset (*Introduction to Statistical Learning*, Chapter 7) records wages and characteristics for 3,000 male workers in the Mid-Atlantic region.

| Variable | Description |
| --- | --- |
| `age` | Worker age (years) |
| `education` | Highest education level (5 ordered categories) |
| `jobclass` | Sector: Industrial or Information |
| `wage` | Annual wage in thousands of dollars |

Download the CSV from the ISL textbook resources page ([statlearning.com](https://www.statlearning.com/resources-second-edition)) and update the URL in the loading cell below.

In [ ]:
# data and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='white')

# sklearn utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

print(f'TensorFlow {tf.__version__}')

In [ ]:
# Wage dataset from ISLR — replace with your Dropbox link or local path
url = 'https://www.dropbox.com/scl/fi/pbeb2oeu8dno5ifwrj396/Wage.csv?rlkey=8s05qnchqc4b47ehqzhz5e3ph&dl=1'

df = pd.read_csv(url)
df = df[['age', 'education', 'jobclass', 'wage']].dropna().reset_index(drop=True)

display(df.head())
display(df.describe().T.round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# raw wage — notice the right skew and high-earning outliers
axes[0].scatter(df['age'], df['wage'], alpha=0.25, s=12, color='steelblue')
axes[0].set_xlabel('Age (years)', fontsize=13)
axes[0].set_ylabel('Wage (thousands of dollars)', fontsize=13)
axes[0].set_title('Raw wage', fontsize=13)

# log wage — more symmetric, easier to model
axes[1].scatter(df['age'], np.log(df['wage']), alpha=0.25, s=12, color='darkorange')
axes[1].set_xlabel('Age (years)', fontsize=13)
axes[1].set_ylabel('Log wage', fontsize=13)
axes[1].set_title('Log wage', fontsize=13)

plt.suptitle('Age vs. Wage — Mid-Atlantic workers', fontsize=14)
plt.tight_layout()
plt.show()

The left panel shows the classic age–wage arc — wages rise with experience, spread widely in middle age, and then level off. But notice the right skew: a long tail of high earners pulls the distribution upward.

The right panel takes **log(wage)**. The distribution is more symmetric, the variance is more stable across age groups, and the scale is more familiar to economists (log wage differences are approximately percentage wage differences). We will use log(wage) as the outcome for the rest of this notebook.

A straight line will still underfit the curved relationship. A degree-4 polynomial *could* fit it — but requires us to decide the degree in advance. A neural network will learn the right shape on its own.

---
## Part 1: Data preparation

Before training any neural network we need to:
1. **Split** into train and test sets so the final evaluation is honest.
2. **Standardize** the inputs — neural networks train much more stably when all inputs are on a comparable scale (the same reason we care about units in OLS).
3. **Standardize the output** — with ReLU activations, all hidden units start with their kinks clustered at zero; if the output scale is large the network wastes most of training just shifting the output bias rather than spreading those kinks. Standardizing log(wage) before training and inverse-transforming predictions solves this.

In Parts 1–3 we use **age alone** as the input and **log(wage)** as the outcome. With one input we can overlay the fitted curve directly on the scatter plot, making every modelling choice immediately visible.

In [ ]:
X_age = df[['age']].values
y     = np.log(df['wage'].values)  # log wage: more symmetric, easier to model

# 80 / 20 train-test split
X_train_age, X_test_age, y_train, y_test = train_test_split(
    X_age, y, test_size=0.2, random_state=1706
)

# standardize inputs — fit on train only, apply to both
scaler_age    = StandardScaler()
X_train_age_s = scaler_age.fit_transform(X_train_age)
X_test_age_s  = scaler_age.transform(X_test_age)

# standardize the output too — without this, all hidden units waste most of training
# moving the output bias toward the mean instead of spreading their kinks
scaler_y    = StandardScaler()
y_train_s   = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
# y_test stays in log-wage scale so MSE is comparable across all models

# fine age grid for plotting smooth fitted curves
age_grid   = np.linspace(df['age'].min(), df['age'].max(), 300).reshape(-1, 1)
age_grid_s = scaler_age.transform(age_grid)

print(f'Train: {len(X_train_age_s)} rows  |  Test: {len(X_test_age_s)} rows')

---
## Part 2: Linear regression as a neural network

A neural network with one input, one output, and **no hidden layers** is identical to linear regression:
$$\hat{\text{wage}} = w \cdot \text{age}_{\text{scaled}} + b.$$

Building this baseline in Keras first makes the step to a "real" neural network concrete — we will literally add one line of code to go from OLS to a nonlinear model.

In [ ]:
# no hidden layers: this is linear regression
linear_model = keras.Sequential([
    layers.Dense(1, input_shape=(1,))  # one weight + one bias
])

linear_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mse')
linear_model.summary()

In [ ]:
class LinearModelTracker(callbacks.Callback):
    def __init__(self, age_grid_s, scaler_y):
        super().__init__()
        self.age_grid_s = age_grid_s
        self.scaler_y = scaler_y
        self.weight_history = []
        self.bias_history = []
        self.prediction_history = []

    def on_epoch_end(self, epoch, logs=None):
        weights, bias = self.model.layers[0].get_weights()
        self.weight_history.append(float(weights[0, 0]))
        self.bias_history.append(float(bias[0]))
        preds = self.model.predict(self.age_grid_s, verbose=0)
        preds = self.scaler_y.inverse_transform(preds).flatten()
        self.prediction_history.append(preds)

linear_tracker = LinearModelTracker(age_grid_s, scaler_y)

history_linear = linear_model.fit(
    X_train_age_s, y_train_s,
    epochs=100,
    validation_split=0.2,
    callbacks=[linear_tracker],
    verbose=0
)

preds_linear = scaler_y.inverse_transform(
    linear_model.predict(X_test_age_s, verbose=0)
).flatten()

linear_epochs = np.arange(1, len(linear_tracker.weight_history) + 1)

test_mse = {}
test_mse['Linear (age only)'] = mean_squared_error(y_test, preds_linear)
print(f"Test MSE — linear model: {test_mse['Linear (age only)']:.5f}")

In [ ]:
# extract the two parameters from the linear model
[[w_std]], [b_std] = linear_model.get_weights()

# de-standardize: model predicts log_wage_std = w*age_std + b
# => d(log_wage)/d(age) = w * std_log_wage / std_age
slope = w_std * float(scaler_y.scale_[0]) / float(scaler_age.scale_[0])

print("Linear neural network — extracted parameters")
print(f"  weight w  (standardized):  {w_std:+.4f}   (age_std → log_wage_std)")
print(f"  bias   b  (standardized):  {b_std:+.4f}")
print()
print(f"  Implied slope: {slope:.4f}  Δ log wage per year of age")
print(f"  ≈ {slope * 100:.2f}% wage increase per additional year")
print()
print("  A linear neural network is OLS — gradient descent converges to the")
print("  same solution as the closed-form normal equations.")

In [ ]:
preds = scaler_y.inverse_transform(
    linear_model.predict(age_grid_s, verbose=0)
).flatten()

# Closed-form OLS on the same data Adam sees.
# Keras validation_split=0.2 takes the last 20% of the array as validation,
# so Adam only trains on the first 80%. Slice identically here so the two
# solutions are comparable.
n_keras_train = int(len(X_train_age_s) * 0.8)
ols_age_model = LinearRegression().fit(
    X_train_age_s[:n_keras_train], y_train_s[:n_keras_train]
)
w_ols = float(ols_age_model.coef_[0])
b_ols = float(ols_age_model.intercept_)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sample_epochs = np.unique(np.linspace(0, len(linear_tracker.prediction_history) - 1, 8, dtype=int))
for idx in sample_epochs:
    alpha = 0.2 + 0.6 * (idx + 1) / len(linear_tracker.prediction_history)
    axes[0].plot(
        age_grid.flatten(),
        linear_tracker.prediction_history[idx],
        color='tomato',
        lw=1.2,
        alpha=alpha
    )

axes[0].scatter(df['age'], np.log(df['wage']), alpha=0.2, s=12, color='steelblue', label='Data')
axes[0].plot(age_grid.flatten(), preds, color='black', lw=2.5, label='Final fit')
axes[0].set_xlabel('Age (years)', fontsize=13)
axes[0].set_ylabel('Log wage', fontsize=13)
axes[0].set_title('How the linear fit evolves across training', fontsize=14)
axes[0].legend(fontsize=12)

axes[1].plot(linear_epochs, linear_tracker.weight_history, color='purple', lw=2, label='Weight on age (GD)')
axes[1].plot(linear_epochs, linear_tracker.bias_history, color='darkorange', lw=2, label='Bias term (GD)')
axes[1].axhline(w_ols, color='purple', linestyle='--', lw=1.8, alpha=0.9, label='Weight (OLS closed form)')
axes[1].axhline(b_ols, color='darkorange', linestyle='--', lw=1.8, alpha=0.9, label='Bias (OLS closed form)')
axes[1].set_xlabel('Epoch', fontsize=13)
axes[1].set_ylabel('Parameter value (standardized scale)', fontsize=13)
axes[1].set_title('Weight and bias over epochs', fontsize=14)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

---
## Part 3: Adding a hidden layer

Now we insert one hidden layer between the input and output. Each unit in the hidden layer applies a **ReLU** activation — it is linear for positive inputs and zero for negative ones. Combining many of these units lets the network approximate curved functions without us specifying polynomial terms.

$$z_k = w_k^{(1)} \cdot \text{age}_{\text{scaled}} + b_k^{(1)}, \qquad a_k = \text{ReLU}(z_k) = \max(0,\, z_k)$$
$$\hat{\text{wage}} = \sum_{k=1}^{K} v_k \, a_k + b^{(2)}$$

Each of the $K$ hidden units creates a "bent line"; the output layer combines them into a smooth, flexible curve.

> **Interactive:** change `N_UNITS` to 8, 32, or 256 and re-run. How does the fitted curve change with more or fewer units?

In [ ]:
N_UNITS = 64  # Interactive: try 8, 32, 128, 256

shallow_model = keras.Sequential([
    layers.Dense(N_UNITS, activation='relu', input_shape=(1,)),  # hidden layer
    layers.Dense(1)                                               # linear output
])

# Try uncommenting one optimizer choice at a time and compare the fitted curve.
# optimizer = keras.optimizers.Adam(learning_rate=0.01)
optimizer = keras.optimizers.Adam(learning_rate=0.0005)
# optimizer = keras.optimizers.Adam(learning_rate=0.00001)
# optimizer = keras.optimizers.SGD(learning_rate=0.01)
# optimizer = keras.optimizers.SGD(learning_rate=0.001, momentum=0.9)
# optimizer = keras.optimizers.RMSprop(learning_rate=0.0001)

shallow_model.compile(optimizer=optimizer, loss='mse')
shallow_model.summary()

In [ ]:
early_stop = callbacks.EarlyStopping(
    monitor='val_loss', patience=30, restore_best_weights=True
)

history_shallow = shallow_model.fit(
    X_train_age_s, y_train_s,
    epochs=500,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=0
)

preds_shallow = scaler_y.inverse_transform(
    shallow_model.predict(X_test_age_s, verbose=0)
).flatten()

test_mse[f'Shallow net, {N_UNITS} units (age only)'] = mean_squared_error(y_test, preds_shallow)
print(f"Test MSE — shallow model: {test_mse[f'Shallow net, {N_UNITS} units (age only)']:.5f}")

In [ ]:
# --- extract weights from the shallow model ---
W1, b1 = shallow_model.layers[0].get_weights()   # shapes: (1, N_UNITS), (N_UNITS,)
W2, b2 = shallow_model.layers[1].get_weights()   # shapes: (N_UNITS, 1), (1,)

# kink location for unit k: where pre-activation = 0
#   W1[0,k] * age_std + b1[k] = 0  =>  age_std = -b1[k] / W1[0,k]
age_std_kinks = -b1 / W1[0, :]
age_kinks     = float(scaler_age.mean_[0]) + float(scaler_age.scale_[0]) * age_std_kinks
out_weights   = W2[:, 0]

age_min, age_max = df['age'].min(), df['age'].max()
in_range = (age_kinks >= age_min) & (age_kinks <= age_max)
print(f"{in_range.sum()} of {N_UNITS} kinks fall within the observed age range ({age_min}–{age_max})\n")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- left: fitted curve with kink locations as vertical lines ---
ax = axes[0]
preds_grid_s = scaler_y.inverse_transform(
    shallow_model.predict(age_grid_s, verbose=0)
).flatten()
ax.scatter(df['age'], np.log(df['wage']), alpha=0.15, s=10, color='steelblue')
ax.plot(age_grid.flatten(), preds_grid_s, color='seagreen', lw=2.5, zorder=3, label='Fitted curve')
for kage, ow in zip(age_kinks[in_range], out_weights[in_range]):
    color = '#d62728' if ow > 0 else '#1f77b4'
    ax.axvline(kage, color=color, alpha=0.3, lw=1, zorder=2)
ax.set_xlabel('Age', fontsize=12)
ax.set_ylabel('Log wage', fontsize=12)
ax.set_title('Kink locations overlaid on fitted curve\n(red = positive output weight, blue = negative)', fontsize=11)
ax.legend(fontsize=11)

# --- right: kink location vs. output weight ---
ax = axes[1]
sc = ax.scatter(age_kinks[in_range], out_weights[in_range],
                c=out_weights[in_range], cmap='RdBu_r', vmin=-max(abs(out_weights)),
                vmax=max(abs(out_weights)), s=55, zorder=3)
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('Kink location (age, years)', fontsize=12)
ax.set_ylabel('Output weight', fontsize=12)
ax.set_title('Each dot = one hidden unit\n(position = where it activates, height = how much it contributes)', fontsize=11)
plt.colorbar(sc, ax=ax, label='Output weight')

plt.suptitle(f'Shallow network: how its {N_UNITS} hidden units shape the fitted curve', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, model, title, color in zip(
    axes,
    [linear_model, shallow_model],
    ['Linear (no hidden layers)', f'Shallow net ({N_UNITS} units, 1 hidden layer)'],
    ['tomato', 'seagreen']
):
    preds = scaler_y.inverse_transform(
        model.predict(age_grid_s, verbose=0)
    ).flatten()
    ax.scatter(df['age'], np.log(df['wage']), alpha=0.2, s=10, color='steelblue')
    ax.plot(age_grid.flatten(), preds, color=color, lw=2.5)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Age', fontsize=12)
    ax.set_ylabel('Log wage', fontsize=12)

plt.suptitle('Linear vs. shallow neural network', fontsize=15)
plt.tight_layout()
plt.show()

# In-Class Exercise #1

Q1: Look at the two plots side by side. How does the shallow network's fitted curve differ from the linear fit? In which parts of the age range is the added flexibility most apparent?

Q2: `EarlyStopping` halts training when the **validation loss** stops improving for `patience` consecutive epochs. Why do we monitor validation loss rather than training loss for this decision? What would happen if we only monitored training loss?

---
## Part 4: Training curves and the bias–variance tradeoff

Every call to `model.fit()` records the loss on the training set and, with `validation_split`, on a held-out portion. These **training curves** are a direct window into the bias–variance tradeoff:

- **Training loss stays high** → the model is underfitting (high bias, too simple).
- **Training loss falls but validation loss stays high or rises** → the model is overfitting (high variance, too flexible).
- The **gap** between the two curves at convergence measures how much variance the model carries.

`EarlyStopping` automates finding the sweet spot by stopping before the validation loss climbs back up.

In [ ]:
def plot_training_curves(history, title='Training curves', max_epochs=None):
    train_loss = history.history['loss']
    val_loss   = history.history['val_loss']
    epochs     = range(1, len(train_loss) + 1)
    if max_epochs:
        epochs, train_loss, val_loss = (
            list(epochs)[:max_epochs], train_loss[:max_epochs], val_loss[:max_epochs]
        )
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(epochs, train_loss, label='Training loss',   color='steelblue')
    ax.plot(epochs, val_loss,   label='Validation loss', color='tomato')
    ax.set_xlabel('Epoch', fontsize=13)
    ax.set_ylabel('MSE', fontsize=13)
    ax.set_title(title, fontsize=14)
    ax.legend(fontsize=12)
    plt.tight_layout()
    plt.show()

plot_training_curves(history_shallow, title=f'Shallow network ({N_UNITS} units) — training vs. validation loss')

# In-Class Exercise #2

Q3: In the training curve above, do you see evidence of overfitting? How many epochs did early stopping allow before halting?

Q4: Re-run Part 3 with `N_UNITS = 256`. How do the training curves change compared to `N_UNITS = 64`? Does the test MSE improve, stay the same, or get worse?

---
## Part 5: Going deeper

A single hidden layer can, in theory, approximate any continuous function given enough units. In practice, **depth** — stacking multiple layers — often learns richer patterns more efficiently than just making one layer very wide.

Intuitively: the first hidden layer learns simple features (gentle bends); the second layer combines those into more complex patterns; and so on — the same hierarchical logic from the lecture.

**One caveat:** depth helps most when the underlying patterns are genuinely hierarchical (edges → shapes → objects in images, for example). For a simple 1D regression like age → log wage, a shallow network often matches or beats a deep one — the curve is smooth and the optimization landscape of a deeper network is harder to navigate. Don't be surprised if the deep model looks *less* flexible here; that is the expected result.

> **Interactive:** start with `N_LAYERS = 2, N_UNITS = 64`. Then try `N_LAYERS = 4` or `N_UNITS = 128`. Watch how the fitted curve and training dynamics change.

In [ ]:
N_LAYERS = 2  # Interactive: try 1, 3, 4
N_UNITS  = 64  # Interactive: try 32, 128, 256

# build the architecture in a loop so N_LAYERS is easy to change
deep_model = keras.Sequential()
deep_model.add(layers.Input(shape=(1,)))
for _ in range(N_LAYERS):
    deep_model.add(layers.Dense(N_UNITS, activation='relu'))
deep_model.add(layers.Dense(1))  # linear output for regression

deep_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0005), loss='mse')
deep_model.summary()

In [ ]:
early_stop_deep = callbacks.EarlyStopping(
    monitor='val_loss', patience=50, restore_best_weights=True
)

history_deep = deep_model.fit(
    X_train_age_s, y_train_s,
    epochs=500,
    validation_split=0.2,
    callbacks=[early_stop_deep],
    verbose=0
)

preds_deep = scaler_y.inverse_transform(
    deep_model.predict(X_test_age_s, verbose=0)
).flatten()

label_deep = f'Deep net ({N_LAYERS}L \u00d7 {N_UNITS}U, age only)'
test_mse[label_deep] = mean_squared_error(y_test, preds_deep)
print(f"Test MSE — deep model: {test_mse[label_deep]:.5f}")

plot_training_curves(history_deep, title=f'Deep network ({N_LAYERS} layers \u00d7 {N_UNITS} units)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

configs = [
    (linear_model,  'Linear (OLS)',                              'tomato'),
    (shallow_model, f'Shallow (1 layer, {N_UNITS} units)',       'seagreen'),
    (deep_model,    f'Deep ({N_LAYERS} layers, {N_UNITS} units)', 'mediumpurple'),
]

for ax, (model, title, color) in zip(axes, configs):
    preds = scaler_y.inverse_transform(
        model.predict(age_grid_s, verbose=0)
    ).flatten()
    ax.scatter(df['age'], np.log(df['wage']), alpha=0.2, s=10, color='steelblue')
    ax.plot(age_grid.flatten(), preds, color=color, lw=2.5)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Age', fontsize=12)
    ax.set_ylabel('Log wage', fontsize=12)

plt.suptitle('How the fit changes as we go deeper', fontsize=15)
plt.tight_layout()
plt.show()

---
## Part 6: Adding more predictors

Age alone explains only part of wage variation. Education level and job sector (Industrial vs. Information) also matter — and we can add them in without changing the model architecture, only the input size.

With multiple inputs we can no longer plot the fitted surface as a simple curve, so we switch to comparing **test MSE** across models and benchmark everything against **OLS**.

In [ ]:
df_enc = pd.get_dummies(df, columns=['education', 'jobclass'], drop_first=False)
feature_cols = [c for c in df_enc.columns if c != 'wage']

X_all = df_enc[feature_cols].values.astype(float)
y_all = np.log(df_enc['wage'].values)  # log wage, consistent with Parts 1–3

X_train_all, X_test_all, y_train_all, y_test_all = train_test_split(
    X_all, y_all, test_size=0.2, random_state=1706
)

scaler_all      = StandardScaler()
X_train_all_s   = scaler_all.fit_transform(X_train_all)
X_test_all_s    = scaler_all.transform(X_test_all)

# standardize y — same scaler fitted in Part 1 (same split, same log-wage values)
y_train_all_s = scaler_y.transform(y_train_all.reshape(-1, 1)).flatten()

print(f'Features ({len(feature_cols)} total): {feature_cols}')

In [ ]:
# OLS baseline with all features
ols = LinearRegression()
ols.fit(X_train_all_s, y_train_all)
test_mse['OLS (all features)'] = mean_squared_error(y_test_all, ols.predict(X_test_all_s))
print(f"OLS test MSE: {test_mse['OLS (all features)']:.5f}")

In [ ]:
full_model = keras.Sequential([
    layers.Input(shape=(X_train_all_s.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)
])
full_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0005), loss='mse')

history_full = full_model.fit(
    X_train_all_s, y_train_all_s,
    epochs=500,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=0
)

preds_full = scaler_y.inverse_transform(
    full_model.predict(X_test_all_s, verbose=0)
).flatten()
test_mse['Neural net (all features)'] = mean_squared_error(y_test_all, preds_full)

plot_training_curves(history_full, title='Neural network — all features')

In [ ]:
# final comparison across all models
comparison = (
    pd.DataFrame({'Model': list(test_mse.keys()), 'Test MSE': list(test_mse.values())})
    .sort_values('Test MSE')
    .reset_index(drop=True)
)
display(comparison.round(5))

# In-Class Exercise #3

Q5: Look at the comparison table. Does the neural network with all features outperform OLS? Is the gap large in practical terms — would it meaningfully change a policy or business decision?

Q6: In Class 10 we used **Lasso** to control model complexity by shrinking coefficients toward zero. A neural network also controls complexity — through architecture choices (layers, units) and early stopping. What is the key difference in *how* each method selects which features to use?

Q7: A policymaker asks you to explain your wage model. Which would you show them — OLS or the neural network — and why? What does your answer depend on?

---
## Summary

| Concept | What you saw today |
| --- | --- |
| Log outcome | Raw wage is right-skewed; log(wage) is more symmetric and standard in labour economics |
| Linear model = 0 hidden layers | `Dense(1)` with no hidden layer is OLS — the same line |
| Hidden layers add nonlinearity | One layer of ReLU units → the curve bends without polynomial terms |
| Bias–variance in training curves | Training loss $\downarrow$ while validation loss rises = overfitting |
| Early stopping | Automates finding the bias–variance sweet spot |
| Width vs. depth | More units → broader patterns; more layers → richer hierarchies |
| More predictors | Education and job class reduce MSE beyond age alone |

The core message: a neural network is not magic — it is a flexible function approximator built from the same linear algebra you already know. What it adds is the ability to learn **which** nonlinearities and interactions matter, rather than requiring you to specify them.